# Task 1：实现 LIF 神经元

目标：模拟**一个神经元**接收恒定电流后，膜电位如何变化、何时发放脉冲。依次运行下方单元格即可。

## 1. LIF 神经元模型

LIF 的本质是：**输入电流改变膜电位，泄漏使膜电位趋向静息值**。发放和重置前，膜电位满足：

$$\tau_m\frac{dV(t)}{dt}=-(V(t)-V_{\mathrm{rest}})+RI(t)$$

其中：

- $V$ 和 $V_{\mathrm{rest}}$：当前膜电位和静息电位；
- $\frac{dV}{dt}$：膜电位随时间变化的速度；
- $-(V-V_{\mathrm{rest}})$：泄漏，使膜电位趋向静息值；
- $RI(t)$：输入电流的作用；
- $\tau_m$：膜时间常数，影响电位变化的快慢。

令 $k$ 为时间步编号，$t_k=k\Delta t$。使用前向 Euler 方法，将连续变化近似为离散更新：

$$V_{k+1}=V_k+\frac{\Delta t}{\tau_m}\left[-(V_k-V_{\mathrm{rest}})+RI_k\right]$$

达到阈值后的脉冲发放、重置和不应期是额外规则。下面的 `LIFNeuron` 在内部适配 `snn.Leaky`，对外使用题目中的物理参数。

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import snntorch as snn
import torch

## 2. 模拟一个神经元

`LIFNeuron` 接受电流（nA），返回膜电位（mV）和脉冲 Tensor。达到阈值时发放并立即重置，随后 2 ms 不应期内保持重置电位、不发放。使用 snnTorch 代理梯度让脉冲在反向传播时可近似求导；不应期仍是离散状态。

每次新序列调用 `reset_state()` 回到静息电位；反向传播跨时间步时应在序列结束后重置状态。

In [ ]:
# 电位 mV、时间 ms、电阻 MΩ、电流 nA；dt 和 tau_m 对应公式中的 Δt 和 τₘ
V_rest, V_reset, V_th = -65.0, -65.0, -50.0
dt, tau_m, R, I = 0.1, 20.0, 10.0, 2.0
refractory_ms, duration_ms = 2.0, 2000.0

In [ ]:
class LIFNeuron(torch.nn.Module):
    def __init__(self, dt, tau_m, R, V_rest, V_reset, V_th, refractory_ms):
        super().__init__()
        if not all(
            np.isfinite(x) for x in (dt, tau_m, R, V_rest, V_reset, V_th, refractory_ms)
        ):
            raise ValueError("LIF parameters must be finite")
        if dt <= 0 or tau_m <= 0 or dt > tau_m or R <= 0:
            raise ValueError("require 0 < dt <= tau_m and R > 0")
        if refractory_ms < 0 or not np.isclose(
            refractory_ms / dt, round(refractory_ms / dt), rtol=0, atol=1e-7
        ):
            raise ValueError("refractory_ms must be a nonnegative multiple of dt")
        if V_reset >= V_th or V_rest >= V_th:
            raise ValueError("rest and reset potentials must be below threshold")
        self.dt, self.tau_m, self.R = dt, tau_m, R
        self.V_rest, self.V_reset = V_rest, V_reset
        self.refractory_steps = round(refractory_ms / dt)
        self.surrogate = snn.surrogate.fast_sigmoid()
        self.neuron = snn.Leaky(
            beta=torch.tensor(self._decay_factor(), dtype=torch.float32),
            threshold=self._relative_voltage(V_th),
            reset_mechanism="zero",
            reset_delay=False,
            spike_grad=self._at_threshold,
        )
        self.register_buffer(
            "U",
            torch.tensor(0.0, dtype=torch.float32),
            persistent=False,
        )
        self.register_buffer(
            "remaining", torch.zeros((), dtype=torch.long), persistent=False
        )

    def _at_threshold(self, voltage_from_threshold):
        # 前向达到阈值即发放；反向使用 snnTorch 的代理梯度。
        return self.surrogate(voltage_from_threshold) + (
            voltage_from_threshold == 0
        ).to(voltage_from_threshold.dtype)

    def _relative_voltage(self, V):
        return V - self.V_rest

    def _decay_factor(self):
        return 1 - self.dt / self.tau_m

    def _input_increment(self, I):
        return self.dt / self.tau_m * self.R * I

    def reset_state(self):
        self.U = torch.zeros_like(self.U)
        self.remaining = torch.zeros_like(self.remaining)
        self.neuron.reset_mem()

    def forward(self, I):
        if not isinstance(I, torch.Tensor):
            I = torch.as_tensor(I, dtype=self.U.dtype, device=self.U.device)
        else:
            I = I.to(dtype=self.U.dtype, device=self.U.device)
        if I.ndim != 0:
            raise ValueError("single-neuron input must be a scalar")
        X = self._input_increment(I)
        spike, U = self.neuron(X, self.U)
        active = self.remaining == 0
        spike = torch.where(active, spike, torch.zeros_like(spike))
        U_reset = self.U.new_tensor(self._relative_voltage(self.V_reset))
        self.U = torch.where(active, U + spike * U_reset, U_reset)
        self.remaining = torch.where(
            active,
            spike.detach().to(torch.long) * self.refractory_steps,
            self.remaining - 1,
        )
        return self.U + self.V_rest, spike

In [ ]:
def simulate(I, duration_ms, dt):
    if not np.isfinite(duration_ms) or duration_ms <= 0:
        raise ValueError("duration_ms must be positive and finite")
    neuron = LIFNeuron(dt, tau_m, R, V_rest, V_reset, V_th, refractory_ms)
    if round(duration_ms / dt) < 1 or not np.isclose(
        duration_ms / dt, round(duration_ms / dt), rtol=0, atol=1e-7
    ):
        raise ValueError("duration_ms must be a multiple of dt")
    voltages, spikes = [], []
    for _ in range(round(duration_ms / dt)):
        V, spike = neuron(I)
        voltages.append(V)
        spikes.append(spike)
    # 只在绘图前将 Tensor 转换为 NumPy。
    return torch.stack(voltages).detach().cpu().numpy(), torch.stack(
        spikes
    ).detach().cpu().numpy()

## 3. 观察一次仿真

给神经元施加恒定电流，展示输入电流、膜电位、脉冲序列和发放率。发放率根据最近 100 ms 内的实际脉冲数计算；不足 100 ms 时按已有时间计算。图中展示前 200 ms，膜电位记录每一步结束后的值，发放后已重置。


In [ ]:
def firing_rate(spikes, dt):
    # 回看最近 100 ms；起始阶段按已有时长计算。
    window_steps = max(1, round(100 / dt))
    cumulative = np.concatenate(([0], np.cumsum(spikes)))
    end = np.arange(1, len(spikes) + 1)
    start = np.maximum(0, end - window_steps)
    return (cumulative[end] - cumulative[start]) / ((end - start) * dt / 1000)

In [ ]:
voltages, spikes = simulate(I=I, duration_ms=duration_ms, dt=dt)
time_ms = (np.arange(len(voltages)) + 1) * dt
rates = firing_rate(spikes, dt)

fig, axes = plt.subplots(4, 1, figsize=(10, 8), sharex=True)
axes[0].plot(time_ms, np.full_like(time_ms, I))
axes[0].set_ylabel("Current (nA)")
axes[1].plot(time_ms, voltages, label="Membrane potential")
axes[1].axhline(V_th, color="red", linestyle="--", label="Threshold")
axes[1].set_ylabel("Voltage (mV)")
axes[1].legend()
axes[2].eventplot(time_ms[spikes.astype(bool)])
axes[2].set(ylabel="Spike", yticks=[])
axes[3].plot(time_ms, rates)
axes[3].set(xlabel="Time (ms)", ylabel="Firing rate (Hz)", xlim=(0, 200))
fig.tight_layout()
plt.show()
print(
    f"脉冲数：{int(spikes.sum())}；平均发放率：{spikes.sum() / (len(spikes) * dt / 1000):.1f} Hz"
)

## 4. 用理论检查实现

默认参数下，临界恒定电流为 $(V_{th}-V_{rest})/R=1.5$ nA。低于临界值时电位达不到阈值；恰好等于临界值时电位渐近接近阈值，在有限时间内也不发放。将上面的 2 nA 改为 1 nA、1.5 nA 再运行，观察脉冲是否消失。

后续可开展不同输入和系统对照实验。
